**Aluno:** Otávio Augusto Reis Nascimento

**Projeto:** SalesInsight

**Professor:** Lucas Ribeiro de Lima

**Criando um dataset fictício com gerador - Opção A**

O gerador abaixo cria propositalmente dados “sujos”, que servirão de matéria-prima para o requisito de limpeza.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.05:
            quantidade = None                    # valor nulo
        if random.random() < 0.04:
            preco = None                         # valor nulo
        if random.random() < 0.06:
            produto = "  " + produto + " "       # espacos extras
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"           # data invalida
        if random.random() < 0.10:
            cliente = random.choice([            # ruido no nome
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                "  " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 200 registros.
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-11-06  Cliente_017  Notebook  Computadores         Norte   
4         5  2025-07-05  Cliente_037    Tablet     Celulares           Sul   

   quantidade  preco_unitario  
0         2.0          102.90  
1         NaN         3204.57  
2         1.0         1939.76  
3         6.0         3864.87  
4        10.0         2008.14  


**1 - Inspecionando os dados**

**Objetivo:** Com o dataset criado, o primeiro passo é realizar a inspeção dos dados, nessa etapa irei utilizar o .shape e verificar as colunas, os tipos de dados, os valores nulos  e os primeiros registros.

In [2]:
print(f"Shape do DataFrame: {df_bruto.shape}")

Shape do DataFrame: (200, 8)


**▶Foi contabilizado 200 linhas e 8 colunas.**

In [3]:
print("Informações sobre as colunas e tipos de dados:")
display(df_bruto.info())

Informações sobre as colunas e tipos de dados:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        200 non-null    int64  
 1   data_venda      200 non-null    object 
 2   cliente         200 non-null    object 
 3   produto         200 non-null    object 
 4   categoria       200 non-null    object 
 5   regiao          200 non-null    object 
 6   quantidade      190 non-null    float64
 7   preco_unitario  196 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 12.6+ KB


None

**▶Tipo de dados encontrados: float, int e object**

In [4]:
print("Contagem de valores nulos por coluna:")
display(df_bruto.isnull().sum())

Contagem de valores nulos por coluna:


,0
id_venda,0
data_venda,0
cliente,0
produto,0
categoria,0
regiao,0
quantidade,10
preco_unitario,4


**▶Total de 14 valores nulos, sendo 10 valores nulos na coluna quantidade e 4 na coluna preco_unitario**

In [5]:
print("Primeiros 5 registros do DataFrame (para visualizar a 'sujeira'):")
display(df_bruto.head())

Primeiros 5 registros do DataFrame (para visualizar a 'sujeira'):


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14


**2 - Limpeza e Tratamento dos Dados**

**Objetivo:** Realizar a limpeza e padronização do dataset, executando cada etapa em células separadas para maior controle e observação do processo. As contagens de registros removidos serão atualizadas em cada passo para o relatório final.

**Primeiro irei importar as bibliotecas, realizar a cópia do DF (como orientado em aula, para não trabalhar no DF original)**

In [6]:
import pandas as pd
import numpy as np
import re

# Criar uma cópia do DataFrame bruto para realizar a limpeza passo a passo
df_limpo = df_bruto.copy()

registros_iniciais = len(df_limpo)
removidos_data = 0
removidos_nulos_criticos = 0
clientes_nao_padronizados_nome = 0

print(f"DataFrame 'df_limpo' criado como cópia de 'df_bruto'. Total inicial de registros: {registros_iniciais}")

DataFrame 'df_limpo' criado como cópia de 'df_bruto'. Total inicial de registros: 200


### 2.1 - Remover espaços extras em colunas de texto

In [7]:
for col in ['cliente', 'produto', 'categoria', 'regiao']:
    if col in df_limpo.columns and df_limpo[col].dtype == 'object':
        df_limpo[col] = df_limpo[col].astype(str).str.strip()
print("  Espaços extras removidos.")

  Espaços extras removidos.


### 2.2 - Converter `data_venda` para datetime e descartar inválidas

In [8]:
pre_date_removal_len = len(df_limpo)
df_limpo['data_venda'] = pd.to_datetime(df_limpo['data_venda'], errors='coerce')
df_limpo.dropna(subset=['data_venda'], inplace=True)
removidos_data = pre_date_removal_len - len(df_limpo)
print(f"  Foram descartados {removidos_data} registros com datas inválidas.")
print(f"  Registros restantes: {len(df_limpo)}")

  Foram descartados 4 registros com datas inválidas.
  Registros restantes: 196


### 2.3 - Remover nulos em `quantidade` e `preco_unitario`

In [9]:
pre_null_removal_len = len(df_limpo)
df_limpo.dropna(subset=['quantidade', 'preco_unitario'], inplace=True)
removidos_nulos_criticos = pre_null_removal_len - len(df_limpo)
print(f"  Foram descartados {removidos_nulos_criticos} registros com nulos em 'quantidade' ou 'preco_unitario'.")
print(f"  Registros restantes: {len(df_limpo)}")

  Foram descartados 13 registros com nulos em 'quantidade' ou 'preco_unitario'.
  Registros restantes: 183


### 2.4 - Ajustar tipos numéricos

In [10]:
df_limpo['quantidade'] = df_limpo['quantidade'].astype(int)
df_limpo['preco_unitario'] = df_limpo['preco_unitario'].astype(float)
print("  Tipos numéricos ajustados.")
display(df_limpo[['quantidade', 'preco_unitario']].dtypes)

  Tipos numéricos ajustados.


,0
quantidade,int64
preco_unitario,float64


### 2.5 - Padronizar nomes de clientes com Regex

In [11]:
def padronizar_cliente_nome(nome_original):
    nome_limpo = re.sub(r'[^a-zA-Z0-9_]', '', str(nome_original)).strip()
    match = re.search(r'cliente_?(\d+)', nome_limpo, re.IGNORECASE)
    if match:
        return f"Cliente_{int(match.group(1)):03d}"
    return f"NAO_PADRAO_{nome_limpo}"

df_limpo['cliente'] = df_limpo['cliente'].apply(padronizar_cliente_nome)
clientes_nao_padronizados_nome = df_limpo[df_limpo['cliente'].str.startswith('NAO_PADRAO_')].shape[0]
if clientes_nao_padronizados_nome > 0:
    print(f"    Atenção: {clientes_nao_padronizados_nome} registros de clientes não puderam ser padronizados para o formato 'Cliente_NNN'.")
print("  Nomes de clientes foram padronizados.")

  Nomes de clientes foram padronizados.


### 2.6 - Relatório Final da Limpeza

In [12]:
registros_finais = len(df_limpo)
total_removidos = registros_iniciais - registros_finais

relatorio_limpeza = {
    'registros_iniciais': registros_iniciais,
    'removidos_data_invalida': removidos_data,
    'removidos_nulos_quantidade_preco': removidos_nulos_criticos,
    'clientes_nao_padronizados_nome': clientes_nao_padronizados_nome,
    'total_registros_removidos': total_removidos,
    'registros_finais': registros_finais
}

print("\n--- Relatório Final de Limpeza ---")
for key, value in relatorio_limpeza.items():
    print(f"{key.replace('_', ' ').capitalize()}: {value}")

print("\n--- Motivo da Remoção dos Registros---")
print(f"Durante o processo de limpeza, foram removidos {removidos_data} registros devido a **datas inválidas** na coluna `data_venda`. \nEstes registros continham valores que não puderam ser convertidos para o formato de data/hora válido, tornando-os inconsistentes para análise temporal.\n")
print(f"Além disso, {removidos_nulos_criticos} registros foram descartados devido à presença de **valores nulos** nas colunas `quantidade` ou `preco_unitario`. \nA ausência de dados nessas colunas é crítica, pois impede o cálculo preciso de vendas e outras métricas financeiras. \nManter registros com esses valores nulos compromete a integridade de qualquer análise futura.")

print("\n--- Informações Finais do DataFrame Limpo ---")
print("\nShape do DataFrame limpo:")
display(df_limpo.shape)

print("\nInformações sobre as colunas e tipos de dados do DataFrame limpo:")
display(df_limpo.info())

print("\nContagem de valores nulos por coluna no DataFrame limpo:")
display(df_limpo.isnull().sum())

print("\nPrimeiros 5 registros do DataFrame limpo:")
display(df_limpo.head())


--- Relatório Final de Limpeza ---
Registros iniciais: 200
Removidos data invalida: 4
Removidos nulos quantidade preco: 13
Clientes nao padronizados nome: 0
Total registros removidos: 17
Registros finais: 183

--- Motivo da Remoção dos Registros---
Durante o processo de limpeza, foram removidos 4 registros devido a **datas inválidas** na coluna `data_venda`. 
Estes registros continham valores que não puderam ser convertidos para o formato de data/hora válido, tornando-os inconsistentes para análise temporal.

Além disso, 13 registros foram descartados devido à presença de **valores nulos** nas colunas `quantidade` ou `preco_unitario`. 
A ausência de dados nessas colunas é crítica, pois impede o cálculo preciso de vendas e outras métricas financeiras. 
Manter registros com esses valores nulos compromete a integridade de qualquer análise futura.

--- Informações Finais do DataFrame Limpo ---

Shape do DataFrame limpo:


(183, 8)


Informações sobre as colunas e tipos de dados do DataFrame limpo:
<class 'pandas.core.frame.DataFrame'>
Index: 183 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_venda        183 non-null    int64         
 1   data_venda      183 non-null    datetime64[ns]
 2   cliente         183 non-null    object        
 3   produto         183 non-null    object        
 4   categoria       183 non-null    object        
 5   regiao          183 non-null    object        
 6   quantidade      183 non-null    int64         
 7   preco_unitario  183 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 12.9+ KB


None


Contagem de valores nulos por coluna no DataFrame limpo:


,0
id_venda,0
data_venda,0
cliente,0
produto,0
categoria,0
regiao,0
quantidade,0
preco_unitario,0



Primeiros 5 registros do DataFrame limpo:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10,2008.14
5,6,2025-08-21,Cliente_041,Headset,Perifericos,Sudeste,2,337.41


**3 - Criação de Colunas Derivadas**

**Objetivo:** Gerar novas colunas no DataFrame `df_limpo` para enriquecer a análise, com base nas transformações condicionais e extração de componentes de data.

**3.1 - Calculando e criando a coluna de 'receita_total' usando do df_limpo**

In [13]:
df_limpo['receita_total'] = df_limpo['quantidade'] * df_limpo['preco_unitario']
print("Coluna 'receita_total' criada.")

Coluna 'receita_total' criada.


Função para verificar os primeiros registros com a nova coluna

In [14]:
display(df_limpo[['quantidade', 'preco_unitario', 'receita_total']].head())

,quantidade,preco_unitario,receita_total
0,2,102.90,205.80
2,1,1939.76,1939.76
3,6,3864.87,23189.22
4,10,2008.14,20081.40
5,2,337.41,674.82


**3.2 - Função para extrair 'mes' e 'ano'**

In [15]:
df_limpo['mes'] = df_limpo['data_venda'].dt.month
df_limpo['ano'] = df_limpo['data_venda'].dt.year
print("Colunas 'mes' e 'ano' extraídas da 'data_venda'.")

Colunas 'mes' e 'ano' extraídas da 'data_venda'.


Função para exibir os primeiros registros com as novas colunas

In [16]:
display(df_limpo[['data_venda', 'mes', 'ano']].head())

,data_venda,mes,ano
0,2025-05-21,5,2025
2,2025-03-23,3,2025
3,2025-11-06,11,2025
4,2025-07-05,7,2025
5,2025-08-21,8,2025


**3.3 - Criação da coluna mes_nome usando um dicionário de mapeamento conforme recomendado no na atividade.**

In [17]:
mes_map = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
df_limpo['mes_nome'] = df_limpo['mes'].map(mes_map)
print("Coluna 'mes_nome' criada.")

Coluna 'mes_nome' criada.


Função para exibir os primeiros registros com a nova coluna

In [18]:
display(df_limpo[['mes', 'mes_nome']].head())

,mes,mes_nome
0,5,Maio
2,3,Março
3,11,Novembro
4,7,Julho
5,8,Agosto


**3.4 - Criando a função 'trimestre'**

In [19]:
df_limpo['trimestre'] = 'Q' + df_limpo['data_venda'].dt.quarter.astype(str)
print("Coluna 'trimestre' criada.")

Coluna 'trimestre' criada.


Função para exibir os primeiros registros com a nova coluna

In [20]:
display(df_limpo[['data_venda', 'trimestre']].head())

,data_venda,trimestre
0,2025-05-21,Q2
2,2025-03-23,Q1
3,2025-11-06,Q4
4,2025-07-05,Q3
5,2025-08-21,Q3


**3.5 - Transformação condicional vetorizada**
Função para criar 'faixa_receita_item' com classificação condicional.

In [21]:
condicoes = [
    df_limpo["receita_total"] < 500,
    (df_limpo["receita_total"] >= 500) & (df_limpo["receita_total"] < 5000),
    df_limpo["receita_total"] >= 5000,
]
faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
df_limpo["faixa_receita_item"] = np.select(condicoes, faixas, default="Nao Classificado")
print("Coluna 'faixa_receita_item' criada.")

Coluna 'faixa_receita_item' criada.


Função para exibir os primeiros registros com a nova coluna

In [22]:
display(df_limpo[['receita_total', 'faixa_receita_item']].head())

,receita_total,faixa_receita_item
0,205.80,Baixo Valor
2,1939.76,Medio Valor
3,23189.22,Alto Valor
4,20081.40,Alto Valor
5,674.82,Medio Valor


**4 - Métricas Agregadas**

**Objetivo:** Calcular e exibir métricas importantes utilizando agrupamentos (`groupby`) para obter insights sobre vendas por diferentes dimensões. As métricas foram calculadas de acordo com Assinatura Suugerida no documento da atividade proposta.

In [23]:
def calcular_metricas(df):
    metricas = {}

    # 4.1 Receita total, quantidade vendida e número de vendas por mês
    # Usando 'mes' e 'ano' para ordenar corretamente e 'mes_nome' para exibição
    metricas['por_mes'] = df.groupby(['ano', 'mes_nome']).agg(
        receita_total=('receita_total', 'sum'),
        quantidade_vendida=('quantidade', 'sum'),
        numero_vendas=('id_venda', 'count')
    ).reset_index().sort_values(by=['ano', 'mes_nome'])

    # 4.2 Receita total por produto (Top 5, em ordem decrescente)
    metricas['top_produtos'] = df.groupby('produto').agg(
        receita_total=('receita_total', 'sum')
    ).reset_index().sort_values(by='receita_total', ascending=False).head(5)

    # 4.3 Receita total por categoria
    metricas['por_categoria'] = df.groupby('categoria').agg(
        receita_total=('receita_total', 'sum')
    ).reset_index().sort_values(by='receita_total', ascending=False)

    # 4.4 Receita total e ticket médio por região
    metricas['por_regiao'] = df.groupby('regiao').agg(
        receita_total=('receita_total', 'sum'),
        # Ticket médio é a receita total dividida pelo número de vendas na região
        ticket_medio=('receita_total', 'mean')
    ).reset_index().sort_values(by='receita_total', ascending=False)

    return metricas

# Calcular as métricas uma vez
metricas_agregadas = calcular_metricas(df_limpo)
print("Métricas agregadas calculadas e armazenadas em 'metricas_agregadas'.")

Métricas agregadas calculadas e armazenadas em 'metricas_agregadas'.


### Visualização Individual das Métricas Agregadas

Agora que todas as métricas foram calculadas pela função `calcular_metricas` e estão armazenadas no dicionário `metricas_agregadas`, podemos visualizá-las individualmente em células separadas.

**Métrica: Receita, Quantidade e Número de Vendas por Mês**

In [24]:
print("\n--- Métrica: Receita, Quantidade e Número de Vendas por Mês ---")
display(metricas_agregadas['por_mes'])


--- Métrica: Receita, Quantidade e Número de Vendas por Mês ---


,ano,mes_nome,receita_total,quantidade_vendida,numero_vendas
0,2025,Abril,80636.64,48,7
1,2025,Agosto,85790.38,72,12
2,2025,Dezembro,104760.44,71,14
3,2025,Fevereiro,73895.74,70,12
4,2025,Janeiro,120866.25,92,15
5,2025,Julho,106667.30,84,14
6,2025,Junho,106534.48,95,15
7,2025,Maio,132080.62,104,19
8,2025,Março,123869.23,98,18
9,2025,Novembro,160297.77,136,23


**Top 5 Produtos por Receita**

In [25]:
print("\n--- Métrica: Top 5 Produtos por Receita ---")
display(metricas_agregadas['top_produtos'])


--- Métrica: Top 5 Produtos por Receita ---


,produto,receita_total
3,Notebook,374174.85
5,Tablet,335335.83
4,Smartphone,303255.94
1,Monitor,169554.67
0,Headset,48368.37


**Receita Total por Categoria**

In [26]:
print("\n--- Métrica: Receita Total por Categoria ---")
display(metricas_agregadas['por_categoria'])


--- Métrica: Receita Total por Categoria ---


,categoria,receita_total
0,Celulares,638591.77
1,Computadores,543729.52
2,Perifericos,108025.01


**Receita Total e Ticket Médio por Região**

In [28]:
print("\n--- Métrica: Receita Total e Ticket Médio por Região ---")
display(metricas_agregadas['por_regiao'])


--- Métrica: Receita Total e Ticket Médio por Região ---


,regiao,receita_total,ticket_medio
1,Nordeste,366321.23,8721.934048
2,Norte,299917.37,7140.889762
0,Centro-Oeste,244586.97,5688.069070
4,Sul,228249.66,6521.418857
3,Sudeste,151271.07,7203.384286


**5 - Segmentação de Clientes por Nível de Gasto**

**Objetivo:** Agrupar os clientes com base no total gasto e classificá-los em segmentos (Bronze, Prata, Ouro) para uma análise mais aprofundada do comportamento de compra.

Função para agrupar por cliente, calcular o total gasto e classificar em seguimentos usando **a lambda** conforme orientado no documento do projeto.

In [29]:
def segmentar_clientes(df):

    # Agrupar por cliente e calcular o total gasto
    gasto_por_cliente = df.groupby('cliente')['receita_total'].sum().reset_index()
    gasto_por_cliente.rename(columns={'receita_total': 'total_gasto'}, inplace=True)

    # Classificar clientes em segmentos usando uma função lambda
    gasto_por_cliente['segmento'] = gasto_por_cliente['total_gasto'].apply(lambda x:
        'Bronze' if x < 5000 else
        ('Prata' if x >= 5000 and x <= 15000 else 'Ouro')
    )
    return gasto_por_cliente

# Aplicar a função para segmentar os clientes
df_segmentacao_clientes = segmentar_clientes(df_limpo)
print("Segmentação de clientes calculada.")

Segmentação de clientes calculada.


### 5.1 - Top 10 Clientes por Gasto Total

In [30]:
print("\n--- Top 10 Clientes por Gasto Total ---")
display(df_segmentacao_clientes.sort_values(by='total_gasto', ascending=False).head(10))


--- Top 10 Clientes por Gasto Total ---


,cliente,total_gasto,segmento
5,Cliente_006,65905.09,Ouro
35,Cliente_036,63433.04,Ouro
38,Cliente_039,59406.90,Ouro
19,Cliente_020,58459.54,Ouro
10,Cliente_011,56391.77,Ouro
33,Cliente_034,52035.60,Ouro
8,Cliente_009,50827.96,Ouro
23,Cliente_024,49146.94,Ouro
36,Cliente_037,46897.17,Ouro
18,Cliente_019,40731.05,Ouro


### 5.2 - Distribuição de Clientes por Segmento

In [31]:
print("\n--- Distribuição de Clientes por Segmento ---")
display(df_segmentacao_clientes['segmento'].value_counts())


--- Distribuição de Clientes por Segmento ---


,count
segmento,
Ouro,34
Prata,10
Bronze,6


**6 - Operações com NumPy**

**Objetivo:** Demonstrar a aplicação de operações vetorizadas, broadcasting, filtragem booleana e funções de agregação do NumPy diretamente sobre os dados para otimizar o desempenho e simplificar o código.

**Neste trecho será aplicada operações NumPy sobre a coluna receita_total.
    Retornando um dicionario com os valores agregados calculados.**

In [32]:
def calcular_estatisticas_numpy(df):

    # Converter a coluna 'receita_total' para um array NumPy
    receitas = df["receita_total"].to_numpy()
    print(f"Array NumPy 'receitas' criado com {len(receitas)} elementos.")

    # Agregações com NumPy
    media_receitas = np.mean(receitas)
    mediana_receitas = np.median(receitas)
    std_receitas = np.std(receitas) # ddof=0 por padrão
    soma_receitas = np.sum(receitas)
    min_receitas = np.min(receitas)
    max_receitas = np.max(receitas)

    # Broadcasting: escalonar o array para o intervalo 0-1
    # Evitar divisão por zero caso max_receitas == min_receitas
    if max_receitas - min_receitas == 0:
        receitas_escalonadas = np.zeros_like(receitas) # Todos os valores são iguais, então escalam para 0
    else:
        receitas_escalonadas = (receitas - min_receitas) / (max_receitas - min_receitas)
    print("Array 'receitas_escalonadas' (0-1) criado usando broadcasting.")

    # Filtragem booleana: vendas acima do valor médio
    vendas_acima_media = receitas[receitas > media_receitas]
    num_vendas_acima_media = len(vendas_acima_media)
    print(f"Filtragem booleana aplicada: {num_vendas_acima_media} vendas acima da média.")

    resultados = {
        "media_receitas": media_receitas,
        "mediana_receitas": mediana_receitas,
        "desvio_padrao_receitas": std_receitas,
        "soma_receitas": soma_receitas,
        "min_receitas": min_receitas,
        "max_receitas": max_receitas,
        "receitas_escalonadas_exemplo": receitas_escalonadas[:5], # Exemplo dos 5 primeiros valores
        "num_vendas_acima_media": num_vendas_acima_media
    }
    return resultados

# Chamar a função e armazenar os resultados
estatisticas_numpy = calcular_estatisticas_numpy(df_limpo)
print("\nEstatísticas NumPy calculadas e armazenadas em 'estatisticas_numpy'.")

Array NumPy 'receitas' criado com 183 elementos.
Array 'receitas_escalonadas' (0-1) criado usando broadcasting.
Filtragem booleana aplicada: 70 vendas acima da média.

Estatísticas NumPy calculadas e armazenadas em 'estatisticas_numpy'.


**6.1 - Exibindo os Resultados das Operações NumPy**

In [33]:
print("\n--- Resultados das Estatísticas NumPy ---")
for key, value in estatisticas_numpy.items():
    print(f"{key.replace('_', ' ').capitalize()}: {value}")


--- Resultados das Estatísticas NumPy ---
Media receitas: 7051.07267759563
Mediana receitas: 3330.66
Desvio padrao receitas: 7809.919232622955
Soma receitas: 1290346.3000000003
Min receitas: 104.19
Max receitas: 38833.200000000004
Receitas escalonadas exemplo: [0.00262361 0.04739522 0.59606558 0.51582031 0.01473392]
Num vendas acima media: 70


**7 - Visualização de Dados com Matplotlib e Seaborn**

**Objetivo:** Criar diversas visualizações para explorar os dados e os insights gerados, utilizando `Matplotlib` e `Seaborn` conforme os requisitos especificados no documento da atividade.

In [34]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração visual global
sns.set_theme(style="whitegrid", palette="viridis") # Usando uma paleta de cores 'viridis'
plt.rcParams["figure.figsize"] = (12, 6) # Tamanho padrão da figura
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

# Criar diretório para salvar os gráficos, se não existir
os.makedirs("outputs/graficos", exist_ok=True)
print("Diretório 'outputs/graficos' verificado/criado.")

Diretório 'outputs/graficos' verificado/criado.


**7.1 - Gráfico de Linha: Receita Total por Mês**

In [36]:
fig, ax = plt.subplots(figsize=(12, 7))

# Recriar mes_map e o mapa reverso para obter o número do mês a partir do nome
mes_map = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
reverse_mes_map = {v: k for k, v in mes_map.items()}

# Criar uma coluna numérica para o mês para garantir a ordenação correta
metricas_agregadas['por_mes']['mes_numero'] = metricas_agregadas['por_mes']['mes_nome'].map(reverse_mes_map)

# Ordenar os dados por ano e pelo novo 'mes_numero'
receita_por_mes_ordenada = metricas_agregadas['por_mes'].sort_values(by=['ano', 'mes_numero'])

sns.lineplot(data=receita_por_mes_ordenada, x="mes_nome", y="receita_total",
             marker="o", linewidth=2, ax=ax, color='skyblue')
ax.set_title("Receita Total por Mês ao Longo do Período")
ax.set_xlabel("Mês")
ax.set_ylabel("Receita Total (R$)")
ax.tick_params(axis='x', rotation=45) # Rotaciona os rótulos do eixo x

plt.tight_layout()
plt.savefig("outputs/graficos/receita_por_mes.png", dpi=150)
plt.close()
print("Gráfico 'receita_por_mes.png' gerado.")

Gráfico 'receita_por_mes.png' gerado.


### 7.2 - Gráfico de Barras: Top 5 Produtos por Receita

In [37]:
# Gráfico de barras: Top 5 Produtos por Receita
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=metricas_agregadas['top_produtos'], y="produto", x="receita_total",
            hue="produto", legend=False, palette="cividis", ax=ax)
ax.set_title("Top 5 Produtos por Receita Total")
ax.set_xlabel("Receita Total (R$)")
ax.set_ylabel("Produto")

plt.tight_layout()
plt.savefig("outputs/graficos/top_produtos.png", dpi=150)
plt.close()
print("Gráfico 'top_produtos.png' gerado.")

Gráfico 'top_produtos.png' gerado.


### 7.3 - Gráfico de Dispersão: Quantidade vs. Receita Total (colorido por Categoria)

In [38]:
# Gráfico de dispersão: Quantidade vs. Receita Total, colorido por Categoria
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=df_limpo, x="quantidade", y="receita_total", hue="categoria",
                palette="cubehelix", s=100, alpha=0.8, ax=ax)
ax.set_title("Quantidade Vendida vs. Receita Total por Item (por Categoria)")
ax.set_xlabel("Quantidade Vendida")
ax.set_ylabel("Receita Total do Item (R$)")
ax.legend(title="Categoria")

plt.tight_layout()
plt.savefig("outputs/graficos/quantidade_vs_receita.png", dpi=150)
plt.close()
print("Gráfico 'quantidade_vs_receita.png' gerado.")

Gráfico 'quantidade_vs_receita.png' gerado.


### 7.4 - Painel Resumo 2x2 com Subplots

In [39]:
# Painel de subplots 2x2
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("SalesInsight PY - Painel Resumo", fontsize=18)

# Subplot 1: Receita Total por Mês
# Usar a coluna 'mes_nome' para o eixo x e garantir a ordem
sns.lineplot(data=receita_por_mes_ordenada, x="mes_nome", y="receita_total",
             marker="o", linewidth=2, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title("Receita Total por Mês")
axes[0, 0].set_xlabel("Mês")
axes[0, 0].set_ylabel("Receita Total (R$)")
axes[0, 0].tick_params(axis='x', rotation=45)

# Subplot 2: Top 5 Produtos por Receita
sns.barplot(data=metricas_agregadas['top_produtos'], y="produto", x="receita_total",
            hue="produto", legend=False, palette="cividis", ax=axes[0, 1])
axes[0, 1].set_title("Top 5 Produtos por Receita")
axes[0, 1].set_xlabel("Receita Total (R$)")
axes[0, 1].set_ylabel("Produto")

# Subplot 3: Quantidade vs. Receita Total (por Categoria)
sns.scatterplot(data=df_limpo, x="quantidade", y="receita_total", hue="categoria",
                palette="cubehelix", s=70, alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title("Quantidade vs. Receita Total")
axes[1, 0].set_xlabel("Quantidade Vendida")
axes[1, 0].set_ylabel("Receita Total do Item (R$)")
axes[1, 0].legend(title="Categoria")

# Subplot 4: Receita Total por Região
sns.barplot(data=metricas_agregadas['por_regiao'], x="regiao", y="receita_total",
            hue="regiao", legend=False, palette="magma", ax=axes[1, 1])
axes[1, 1].set_title("Receita Total por Região")
axes[1, 1].set_xlabel("Região")
axes[1, 1].set_ylabel("Receita Total (R$)")
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajusta o layout para evitar sobreposição, deixando espaço para o suptitle
plt.savefig("outputs/graficos/painel_resumo.png", dpi=150)
plt.close()
print("Gráfico 'painel_resumo.png' gerado.")

Gráfico 'painel_resumo.png' gerado.


Todos os gráficos são gerados e envidados para a pasta **outputs/graficos**

## 8 - Reorganização do Código: Funções Reutilizáveis, Funções de Ordem Superior e Classes

**Objetivo:** Consolidar o fluxo de análise em funções reutilizáveis, demonstrar o uso de funções de ordem superior e encapsular todo o processo em uma classe, conforme as melhores práticas de organização de código.

### 8.1 - Função de Ordem Superior: `processar_coluna`

Esta função aceita outra função como argumento e a aplica a uma coluna do DataFrame, demonstrando flexibilidade e reutilização de código.

In [40]:
def processar_coluna(df, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma funcao de transformacao a uma coluna do DataFrame.
    Demonstra o uso de funcoes como argumento.

    Args:
        df (pd.DataFrame): O DataFrame a ser processado.
        coluna (str): O nome da coluna a ser transformada.
        funcao_transformacao (callable): A função a ser aplicada à coluna.
        nome_saida (str, optional): O nome da nova coluna criada.
                                   Se None, será '{coluna}_transformado'.

    Returns:
        pd.DataFrame: O DataFrame com a nova coluna adicionada.
    """
    nome_saida = nome_saida or f"{coluna}_transformado"
    df[nome_saida] = df[coluna].apply(funcao_transformacao)
    return df

print("Função `processar_coluna` definida.")

Função `processar_coluna` definida.


### 8.2 - Funções Auxiliares para a Classe `AnalisadorDeVendas`

Para que a classe possa reutilizar as lógicas já desenvolvidas, vamos encapsular as etapas de limpeza e criação de colunas em funções únicas.

In [41]:
def limpar_dados(df_bruto_copy):
    """
    Realiza todas as etapas de limpeza e padronização dos dados.
    Retorna o DataFrame limpo e um relatório de limpeza.
    """
    df_limpo_func = df_bruto_copy.copy()
    registros_iniciais_func = len(df_limpo_func)
    removidos_data_func = 0
    removidos_nulos_criticos_func = 0
    clientes_nao_padronizados_nome_func = 0

    # 2.1 - Remover espaços extras em colunas de texto
    for col in ['cliente', 'produto', 'categoria', 'regiao']:
        if col in df_limpo_func.columns and df_limpo_func[col].dtype == 'object':
            df_limpo_func[col] = df_limpo_func[col].astype(str).str.strip()

    # 2.2 - Converter data_venda para datetime e descartar inválidas
    pre_date_removal_len = len(df_limpo_func)
    df_limpo_func['data_venda'] = pd.to_datetime(df_limpo_func['data_venda'], errors='coerce')
    df_limpo_func.dropna(subset=['data_venda'], inplace=True)
    removidos_data_func = pre_date_removal_len - len(df_limpo_func)

    # 2.3 - Remover nulos em `quantidade` e `preco_unitario`
    pre_null_removal_len = len(df_limpo_func)
    df_limpo_func.dropna(subset=['quantidade', 'preco_unitario'], inplace=True)
    removidos_nulos_criticos_func = pre_null_removal_len - len(df_limpo_func)

    # 2.4 - Ajustar tipos numéricos
    df_limpo_func['quantidade'] = df_limpo_func['quantidade'].astype(int)
    df_limpo_func['preco_unitario'] = df_limpo_func['preco_unitario'].astype(float)

    # 2.5 - Padronizar nomes de clientes com Regex (reaproveita a função existente)
    # padronizar_cliente_nome deve ser definida no escopo global ou importada
    df_limpo_func['cliente'] = df_limpo_func['cliente'].apply(padronizar_cliente_nome)
    clientes_nao_padronizados_nome_func = df_limpo_func[df_limpo_func['cliente'].str.startswith('NAO_PADRAO_')].shape[0]

    registros_finais_func = len(df_limpo_func)
    total_removidos_func = registros_iniciais_func - registros_finais_func

    relatorio_limpeza_func = {
        'registros_iniciais': registros_iniciais_func,
        'removidos_data_invalida': removidos_data_func,
        'removidos_nulos_quantidade_preco': removidos_nulos_criticos_func,
        'clientes_nao_padronizados_nome': clientes_nao_padronizados_nome_func,
        'total_registros_removidos': total_removidos_func,
        'registros_finais': registros_finais_func
    }

    return df_limpo_func, relatorio_limpeza_func

print("Função `limpar_dados` definida.")

Função `limpar_dados` definida.


In [42]:
def criar_colunas_derivadas(df_func):
    """
    Cria as colunas derivadas necessárias para a análise.
    Retorna o DataFrame com as novas colunas.
    """
    df_copy = df_func.copy()

    # 3.1 - Calculando e criando a coluna de 'receita_total'
    df_copy['receita_total'] = df_copy['quantidade'] * df_copy['preco_unitario']

    # 3.2 - Função para extrair 'mes' e 'ano'
    df_copy['mes'] = df_copy['data_venda'].dt.month
    df_copy['ano'] = df_copy['data_venda'].dt.year

    # 3.3 - Criação da coluna mes_nome usando um dicionário de mapeamento
    # mes_map deve ser definida no escopo global ou passada como argumento,
    # para este exemplo, assumimos que mes_map está disponível globalmente.
    df_copy['mes_nome'] = df_copy['mes'].map(mes_map)

    # 3.4 - Criando a função 'trimestre'
    df_copy['trimestre'] = 'Q' + df_copy['data_venda'].dt.quarter.astype(str)

    # 3.5 - Transformação condicional vetorizada: 'faixa_receita_item'
    condicoes = [
        df_copy["receita_total"] < 500,
        (df_copy["receita_total"] >= 500) & (df_copy["receita_total"] < 5000),
        df_copy["receita_total"] >= 5000,
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
    df_copy["faixa_receita_item"] = np.select(condicoes, faixas, default="Nao Classificado")

    return df_copy

print("Função `criar_colunas_derivadas` definida.")

Função `criar_colunas_derivadas` definida.


### 8.3 - Classe `AnalisadorDeVendas`

Esta classe encapsula todo o fluxo de análise, desde o carregamento dos dados até a visualização e resumo, reutilizando as funções definidas anteriormente.

In [62]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json # Importar json aqui
import numpy as np # Importar numpy para tratar tipos específicos

class AnalisadorDeVendas:
    """Encapsula o fluxo de analise dos dados de vendas."""
    def __init__(self, caminho_arquivo):
        # Nao chamar super().__init__() aqui se nao ha heranca de outra classe na base
        self.caminho_arquivo = caminho_arquivo
        self.df_bruto = None
        self.df_limpo = None
        self.metricas = {}
        self.df_segmentacao_clientes = None
        self.relatorio_limpeza = {}
        self.estatisticas_numpy = {}
        self.configurar_visualizacao()

    def configurar_visualizacao(self):
        sns.set_theme(style="whitegrid", palette="viridis")
        plt.rcParams["figure.figsize"] = (12, 6)
        plt.rcParams["axes.titlesize"] = 14
        plt.rcParams["axes.labelsize"] = 12
        plt.rcParams["xtick.labelsize"] = 10
        plt.rcParams["ytick.labelsize"] = 10
        os.makedirs("outputs/graficos", exist_ok=True)
        os.makedirs("outputs", exist_ok=True) # Garantir a criação da pasta 'outputs'

    def carregar(self):
        """Le o CSV e guarda o DataFrame bruto."""
        self.df_bruto = pd.read_csv(self.caminho_arquivo)
        print(f"[Analisador] {len(self.df_bruto)} registros brutos lidos.")

    def limpar(self):
        """Limpa os dados reaproveitando limpar_dados()."""
        if self.df_bruto is None:
            print("[Analisador] Erro: DataFrame bruto não carregado. Chame 'carregar()' primeiro.")
            return
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto.copy())
        print(f"[Analisador] {len(self.df_limpo)} registros limpos após processamento.")

    def transformar(self):
        """
        Cria as colunas derivadas e demonstra a função de ordem superior `processar_coluna`.}
        """
        if self.df_limpo is None:
            print("[Analisador] Erro: DataFrame limpo não disponível. Chame 'limpar()' primeiro.")
            return
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)
        print("[Analisador] Colunas derivadas criadas.")

        # Demonstração de processar_coluna
        self.df_limpo = processar_coluna(
            self.df_limpo, "receita_total",
            lambda x: round(x / 1000, 2),
            nome_saida="receita_em_milhares"
        )
        self.df_limpo = processar_coluna(
            self.df_limpo, "quantidade",
            lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
            nome_saida="perfil_volume"
        )
        print("[Analisador] Colunas 'receita_em_milhares' e 'perfil_volume' criadas usando `processar_coluna`.")


    def analisar(self):
        """
        Calcula metricas, segmentacao e operacoes NumPy.
        Reaproveita calcular_metricas(), segmentar_clientes() e calcular_estatisticas_numpy().
        """
        if self.df_limpo is None:
            print("[Analisador] Erro: DataFrame limpo não disponível. Chame 'limpar()' e 'transformar()' primeiro.")
            return
        self.metricas = calcular_metricas(self.df_limpo)
        print("[Analisador] Métricas agregadas calculadas.")

        # Adicionar 'mes_numero' aqui para que esteja disponível para projeção e visualização
        # mes_map é esperado estar disponível globalmente ou ser passado para a classe
        reverse_mes_map = {v: k for k, v in mes_map.items()} # Define reverse map here
        self.metricas['por_mes']['mes_numero'] = self.metricas['por_mes']['mes_nome'].map(reverse_mes_map)
        # DEBUG: Further inspection
        if self.metricas['por_mes']['mes_numero'].isnull().any():
            print("[DEBUG] NaN values found in mes_numero after mapping in AnalisadorDeVendas.analisar.")
            problematic_names = self.metricas['por_mes'][self.metricas['por_mes']['mes_numero'].isnull()]['mes_nome'].unique()
            print(f"[DEBUG] Problematic mes_nome values (AnalisadorDeVendas.analisar): {problematic_names}")
            print(f"[DEBUG] Keys in reverse_mes_map (AnalisadorDeVendas.analisar): {list(reverse_mes_map.keys())}")

        self.df_segmentacao_clientes = segmentar_clientes(self.df_limpo)
        print("[Analisador] Clientes segmentados.")

        self.estatisticas_numpy = calcular_estatisticas_numpy(self.df_limpo)
        print("[Analisador] Estatísticas NumPy calculadas.")

    def visualizar(self):
        """
        Gera e exporta as figuras.
        Reaproveita a lógica de visualização previamente criada, adaptando para usar atributos da classe.
        """
        if not self.metricas or self.df_segmentacao_clientes is None or self.df_limpo is None:
            print("[Analisador] Erro: Dados para visualização não disponíveis. Chame 'analisar()' primeiro.")
            return

        print("[Analisador] Gerando gráficos...")

        # 1 - receita_por_mes.png
        fig, ax = plt.subplots(figsize=(12, 7))
        # 'mes_numero' já deve ter sido criado em analisar()
        receita_por_mes_ordenada_class = self.metricas['por_mes'].sort_values(by=['ano', 'mes_numero'])

        sns.lineplot(data=receita_por_mes_ordenada_class, x="mes_nome", y="receita_total",
                     marker="o", linewidth=2, ax=ax, color='skyblue')
        ax.set_title("Receita Total por Mês ao Longo do Período")
        ax.set_xlabel("Mês")
        ax.set_ylabel("Receita Total (R$)")
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.savefig("outputs/graficos/receita_por_mes.png", dpi=150)
        plt.close()
        print("  Gráfico 'receita_por_mes.png' gerado.")

        # 2 - top_produtos.png
        fig, ax = plt.subplots(figsize=(12, 7))
        sns.barplot(data=self.metricas['top_produtos'], y="produto", x="receita_total",
                    hue="produto", legend=False, palette="cividis", ax=ax)
        ax.set_title("Top 5 Produtos por Receita Total")
        ax.set_xlabel("Receita Total (R$)")
        ax.set_ylabel("Produto")
        plt.tight_layout()
        plt.savefig("outputs/graficos/top_produtos.png", dpi=150)
        plt.close()
        print("  Gráfico 'top_produtos.png' gerado.")

        # 3 - quantidade_vs_receita.png
        fig, ax = plt.subplots(figsize=(12, 7))
        sns.scatterplot(data=self.df_limpo, x="quantidade", y="receita_total", hue="categoria",
                        palette="cubehelix", s=100, alpha=0.8, ax=ax)
        ax.set_title("Quantidade Vendida vs. Receita Total por Item (por Categoria)")
        ax.set_xlabel("Quantidade Vendida")
        ax.set_ylabel("Receita Total do Item (R$)")
        ax.legend(title="Categoria")
        plt.tight_layout()
        plt.savefig("outputs/graficos/quantidade_vs_receita.png", dpi=150)
        plt.close()
        print("  Gráfico 'quantidade_vs_receita.png' gerado.")

        # 4 - painel resumo.png
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle("SalesInsight PY - Painel Resumo", fontsize=18)

        sns.lineplot(data=receita_por_mes_ordenada_class, x="mes_nome", y="receita_total",
                     marker="o", linewidth=2, ax=axes[0, 0], color='skyblue')
        axes[0, 0].set_title("Receita Total por Mês")
        axes[0, 0].set_xlabel("Mês")
        axes[0, 0].set_ylabel("Receita Total (R$)")
        axes[0, 0].tick_params(axis='x', rotation=45)

        sns.barplot(data=self.metricas['top_produtos'], y="produto", x="receita_total",
                    hue="produto", legend=False, palette="cividis", ax=axes[0, 1])
        axes[0, 1].set_title("Top 5 Produtos por Receita")
        axes[0, 1].set_xlabel("Receita Total (R$)")
        axes[0, 1].set_ylabel("Produto")

        sns.scatterplot(data=self.df_limpo, x="quantidade", y="receita_total", hue="categoria",
                        palette="cubehelix", s=70, alpha=0.7, ax=axes[1, 0])
        axes[1, 0].set_title("Quantidade vs. Receita Total")
        axes[1, 0].set_xlabel("Quantidade Vendida")
        axes[1, 0].set_ylabel("Receita Total do Item (R$)")
        axes[1, 0].legend(title="Categoria")

        sns.barplot(data=self.metricas['por_regiao'], x="regiao", y="receita_total",
                    hue="regiao", legend=False, palette="magma", ax=axes[1, 1])
        axes[1, 1].set_title("Receita Total por Região")
        axes[1, 1].set_xlabel("Região")
        axes[1, 1].set_ylabel("Receita Total (R$)")
        axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig("outputs/graficos/painel_resumo.png", dpi=150)
        plt.close()
        print("  Gráfico 'painel_resumo.png' gerado.")

        print("[Analisador] Todos os gráficos foram gerados e salvos em 'outputs/graficos'.")

    def resumo(self):
        """
        Imprime um resumo executivo do que foi processado.
        """
        print("\n" + "="*40)
        print("         RESUMO EXECUTIVO DA ANÁLISE")
        print("="*40)

        if self.relatorio_limpeza:
            print("\n--- Relatório de Limpeza ---")
            for key, value in self.relatorio_limpeza.items():
                print(f"{key.replace('_', ' ').capitalize()}: {value}")

        if self.metricas:
            print("\n--- Principais Métricas ---")
            print(f"Receita Total Geral: R$ {self.metricas['por_mes']['receita_total'].sum():,.2f}")
            print(f"Produto mais vendido (receita): {self.metricas['top_produtos'].iloc[0]['produto']} (R$ {self.metricas['top_produtos'].iloc[0]['receita_total']:,.2f})")
            print(f"Região com maior receita: {self.metricas['por_regiao'].iloc[0]['regiao']} (R$ {self.metricas['por_regiao'].iloc[0]['receita_total']:,.2f})")

        if self.df_segmentacao_clientes is not None:
            print("\n--- Segmentação de Clientes ---")
            display(self.df_segmentacao_clientes['segmento'].value_counts())

        if self.estatisticas_numpy:
            print("\n--- Estatísticas NumPy da Receita ---")
            print(f"Média: R$ {self.estatisticas_numpy['media_receitas']:,.2f}")
            print(f"Mediana: R$ {self.estatisticas_numpy['mediana_receitas']:,.2f}")
            print(f"Desvio Padrão: R$ {self.estatisticas_numpy['desvio_padrao_receitas']:,.2f}")
            print(f"Número de vendas acima da média: {self.estatisticas_numpy['num_vendas_acima_media']}")

        print("\n" + "="*40)
        print("         FIM DO RESUMO")
        print("="*40)

    def exportar_resultados(self):
        """Exporta os resultados do projeto em CSV e JSON."""
        if not self.metricas or self.df_segmentacao_clientes is None or not self.estatisticas_numpy:
            print("[Analisador] Erro: Dados para exportação não disponíveis. Chame 'analisar()' primeiro.")
            return

        print("[Analisador] Exportando resultados...")

        # Escrever as métricas por mês em CSV
        self.metricas["por_mes"].to_csv("outputs/metricas_por_mes.csv", index=False, encoding="utf-8-sig")
        print("  'outputs/metricas_por_mes.csv' gerado.")

        # Escrever a segmentação de clientes em CSV
        self.df_segmentacao_clientes.to_csv("outputs/segmentacao_clientes.csv", index=False, encoding="utf-8-sig")
        print("  'outputs/segmentacao_clientes.csv' gerado.")

        # Preparar estatísticas para JSON (converter tipos NumPy)
        serializavel = {}
        for k, v in self.estatisticas_numpy.items():
            if isinstance(v, (np.float64, np.float32, np.float16)):
                serializavel[k] = round(float(v), 2)
            elif isinstance(v, np.ndarray):
                serializavel[k] = v.tolist() # Converte array NumPy para lista Python
            else:
                serializavel[k] = v # Mantém outros tipos (int, etc.)

        # Escrever as estatísticas gerais em JSON
        caminho_json = "outputs/estatisticas_gerais.json"
        with open(caminho_json, "w", encoding="utf-8") as f:
            json.dump(serializavel, f, indent=4, ensure_ascii=False)
        print(f"  '{caminho_json}' gerado.")

        # Ler de volta o JSON gravado e exibir o conteúdo
        with open(caminho_json, "r", encoding="utf-8") as f:
            conferencia = json.load(f)
        print(f"[Analisador] JSON gravado e lido de volta: {conferencia}")
        print("[Analisador] Exportação de resultados concluída.")

### 8.4 - Demonstração da Classe `AnalisadorDeVendas`

Agora vamos executar todo o fluxo de análise usando uma instância da classe `AnalisadorDeVendas`.

In [46]:
# Instanciar a classe
analisador = AnalisadorDeVendas(caminho_arquivo="vendas.csv")

# Executar o fluxo completo
analisador.carregar()
analisador.limpar()
analisador.transformar()
analisador.analisar()
analisador.visualizar()
analisador.resumo()
analisador.exportar_resultados() # Chamada para exportar os resultados

print("\nDemonstração da classe `AnalisadorDeVendas` concluída.")

[Analisador] 200 registros brutos lidos.
[Analisador] 183 registros limpos após processamento.
[Analisador] Colunas derivadas criadas.
[Analisador] Colunas 'receita_em_milhares' e 'perfil_volume' criadas usando `processar_coluna`.
[Analisador] Métricas agregadas calculadas.
[Analisador] Clientes segmentados.
Array NumPy 'receitas' criado com 183 elementos.
Array 'receitas_escalonadas' (0-1) criado usando broadcasting.
Filtragem booleana aplicada: 70 vendas acima da média.
[Analisador] Estatísticas NumPy calculadas.
[Analisador] Gerando gráficos...
  Gráfico 'receita_por_mes.png' gerado.
  Gráfico 'top_produtos.png' gerado.
  Gráfico 'quantidade_vs_receita.png' gerado.
  Gráfico 'painel_resumo.png' gerado.
[Analisador] Todos os gráficos foram gerados e salvos em 'outputs/graficos'.

         RESUMO EXECUTIVO DA ANÁLISE

--- Relatório de Limpeza ---
Registros iniciais: 200
Removidos data invalida: 4
Removidos nulos quantidade preco: 13
Clientes nao padronizados nome: 0
Total registros re

,count
segmento,
Ouro,34
Prata,10
Bronze,6



--- Estatísticas NumPy da Receita ---
Média: R$ 7,051.07
Mediana: R$ 3,330.66
Desvio Padrão: R$ 7,809.92
Número de vendas acima da média: 70

         FIM DO RESUMO
[Analisador] Exportando resultados...
  'outputs/metricas_por_mes.csv' gerado.
  'outputs/segmentacao_clientes.csv' gerado.
  'outputs/estatisticas_gerais.json' gerado.
[Analisador] JSON gravado e lido de volta: {'media_receitas': 7051.07, 'mediana_receitas': 3330.66, 'desvio_padrao_receitas': 7809.92, 'soma_receitas': 1290346.3, 'min_receitas': 104.19, 'max_receitas': 38833.2, 'receitas_escalonadas_exemplo': [0.0026236147012278395, 0.04739522130826478, 0.5960655849452388, 0.5158203114409586, 0.014733916513745124], 'num_vendas_acima_media': 70}
[Analisador] Exportação de resultados concluída.

Demonstração da classe `AnalisadorDeVendas` concluída.


**9 - Ponto de Entrada: Fluxo Completo do SalesInsight PY**

Vamos agora unificar todo o fluxo de análise em uma função `main()` e utilizar o bloco `if __name__ == "__main__":` para executar o projeto de ponta a ponta. Isso garante a execução ordenada de todas as etapas: geração do dataset (se necessário), inspeção, limpeza, transformação, análise, visualização, resumo e exportação de resultados.

In [47]:
def inspecionar_dados(df):
    """Realiza e exibe a inspeção inicial de um DataFrame."""
    print("\n" + "-"*40)
    print("         INSPEÇÃO INICIAL DOS DADOS")
    print("-"*40)

    print(f"Shape do DataFrame: {df.shape}")
    print("\nInformações sobre as colunas e tipos de dados:")
    df.info()
    print("\nContagem de valores nulos por coluna:")
    display(df.isnull().sum())
    print("\nPrimeiros 5 registros do DataFrame:")
    display(df.head())
    print("-"*40 + "\n")

print("Função `inspecionar_dados` definida.")

Função `inspecionar_dados` definida.


In [48]:
def main():
    """Executa o fluxo completo do SalesInsight PY."""
    print("=" * 60)
    print("          SALESINSIGHT PY - Análise de Dados de Vendas")
    print("=" * 60)

    # Etapa 0 - garantir a existência do dataset
    if not os.path.exists("vendas.csv"):
        print("[MAIN] 'vendas.csv' não encontrado. Gerando novo dataset...")
        # A função gerar_dataset_vendas é definida no início do notebook
        gerar_dataset_vendas().to_csv("vendas.csv", index=False)
        print("[MAIN] 'vendas.csv' gerado com sucesso.")

    # Etapas 1 a 6 - fluxo pela classe AnalisadorDeVendas
    analisador = AnalisadorDeVendas("vendas.csv")
    analisador.carregar()
    inspecionar_dados(analisador.df_bruto) # Usar a função de inspeção
    analisador.limpar()
    analisador.transformar()
    analisador.analisar()
    analisador.visualizar()
    analisador.resumo()
    analisador.exportar_resultados()

    print("\n[CONCLUÍDO] Fluxo finalizado com sucesso.")
    print("=" * 60)

if __name__ == "__main__":
    main()


          SALESINSIGHT PY - Análise de Dados de Vendas
[Analisador] 200 registros brutos lidos.

----------------------------------------
         INSPEÇÃO INICIAL DOS DADOS
----------------------------------------
Shape do DataFrame: (200, 8)

Informações sobre as colunas e tipos de dados:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        200 non-null    int64  
 1   data_venda      200 non-null    object 
 2   cliente         200 non-null    object 
 3   produto         200 non-null    object 
 4   categoria       200 non-null    object 
 5   regiao          200 non-null    object 
 6   quantidade      190 non-null    float64
 7   preco_unitario  196 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 12.6+ KB

Contagem de valores nulos por coluna:


,0
id_venda,0
data_venda,0
cliente,0
produto,0
categoria,0
regiao,0
quantidade,10
preco_unitario,4



Primeiros 5 registros do DataFrame:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14


----------------------------------------

[Analisador] 183 registros limpos após processamento.
[Analisador] Colunas derivadas criadas.
[Analisador] Colunas 'receita_em_milhares' e 'perfil_volume' criadas usando `processar_coluna`.
[Analisador] Métricas agregadas calculadas.
[Analisador] Clientes segmentados.
Array NumPy 'receitas' criado com 183 elementos.
Array 'receitas_escalonadas' (0-1) criado usando broadcasting.
Filtragem booleana aplicada: 70 vendas acima da média.
[Analisador] Estatísticas NumPy calculadas.
[Analisador] Gerando gráficos...
  Gráfico 'receita_por_mes.png' gerado.
  Gráfico 'top_produtos.png' gerado.
  Gráfico 'quantidade_vs_receita.png' gerado.
  Gráfico 'painel_resumo.png' gerado.
[Analisador] Todos os gráficos foram gerados e salvos em 'outputs/graficos'.

         RESUMO EXECUTIVO DA ANÁLISE

--- Relatório de Limpeza ---
Registros iniciais: 200
Removidos data invalida: 4
Removidos nulos quantidade preco: 13
Clientes nao padronizados nome: 0
Total registros r

,count
segmento,
Ouro,34
Prata,10
Bronze,6



--- Estatísticas NumPy da Receita ---
Média: R$ 7,051.07
Mediana: R$ 3,330.66
Desvio Padrão: R$ 7,809.92
Número de vendas acima da média: 70

         FIM DO RESUMO
[Analisador] Exportando resultados...
  'outputs/metricas_por_mes.csv' gerado.
  'outputs/segmentacao_clientes.csv' gerado.
  'outputs/estatisticas_gerais.json' gerado.
[Analisador] JSON gravado e lido de volta: {'media_receitas': 7051.07, 'mediana_receitas': 3330.66, 'desvio_padrao_receitas': 7809.92, 'soma_receitas': 1290346.3, 'min_receitas': 104.19, 'max_receitas': 38833.2, 'receitas_escalonadas_exemplo': [0.0026236147012278395, 0.04739522130826478, 0.5960655849452388, 0.5158203114409586, 0.014733916513745124], 'num_vendas_acima_media': 70}
[Analisador] Exportação de resultados concluída.

[CONCLUÍDO] Fluxo finalizado com sucesso.


## 10 - Funcionalidades Avançadas: Herança, Projeção, Enriquecimento e Análises Estatísticas

Agora, vamos estender a funcionalidade do `SalesInsight PY` com recursos mais avançados, demonstrando herança de classes, projeção de tendências, enriquecimento de dados, gráficos adicionais e cálculos estatísticos avançados (percentis).

### 10.1 - Dados Auxiliares: Metas por Região

Para demonstrar o enriquecimento do dataset, vamos criar um DataFrame simples com metas de receita por região.

In [57]:
def gerar_metas_regiao():
    """Gera um DataFrame auxiliar com metas de receita por região."""
    data_metas = {
        'regiao': ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte'],
        'meta_receita': [350000, 180000, 300000, 100000, 80000]
    }
    return pd.DataFrame(data_metas)

df_metas = gerar_metas_regiao()
print("DataFrame de metas por região gerado:")
display(df_metas)


DataFrame de metas por região gerado:


,regiao,meta_receita
0,Sudeste,350000
1,Sul,180000
2,Nordeste,300000
3,Centro-Oeste,100000
4,Norte,80000


### 10.2 - Subclasse `AnalisadorComProjecao`

Esta subclasse herda de `AnalisadorDeVendas` e adiciona os métodos para projeção de receita, enriquecimento de dados, cálculos de percentis e visualizações adicionais.

In [64]:
class AnalisadorComProjecao(AnalisadorDeVendas):
    """Estende AnalisadorDeVendas com projeção de tendência, percentis e gráficos extras."""
    def __init__(self, caminho_arquivo):
        super().__init__(caminho_arquivo)
        self.projecao_receita = {}
        self.df_enriquecido_metas = None
        self.percentis_receita = {}

    def projetar_tendencia(self, meses_a_projetar=2, janela_media_movel=3):
        """Estima a receita dos próximos meses por média móvel dos últimos meses."""
        if not self.metricas or 'por_mes' not in self.metricas:
            print("[AnalisadorComProjecao] Erro: Métricas mensais não disponíveis para projeção.")
            return

        df_mensal = self.metricas['por_mes'].copy()

        # DEBUG: Add more checks right before to_datetime
        if 'mes_numero' not in df_mensal.columns:
            print("[DEBUG] 'mes_numero' column is missing in df_mensal before projection.")
            print(f"[DEBUG] Columns in df_mensal: {df_mensal.columns.tolist()}")
            # Attempt to re-create mes_numero if missing, though it should be from super().analisar()
            if 'mes_nome' in df_mensal.columns:
                reverse_mes_map = {v: k for k, v in mes_map.items()}
                df_mensal['mes_numero'] = df_mensal['mes_nome'].map(reverse_mes_map)
                if df_mensal['mes_numero'].isnull().any():
                    print("[DEBUG] 'mes_numero' still has NaNs after re-mapping in projetar_tendencia.")
                    problematic_names_proj = df_mensal[df_mensal['mes_numero'].isnull()]['mes_nome'].unique()
                    print(f"[DEBUG] Problematic mes_nome values in projetar_tendencia: {problematic_names_proj}")
            else:
                print("[DEBUG] 'mes_nome' also missing, cannot re-create 'mes_numero'.")
                return # Exit if critical columns are missing

        print(f"[DEBUG] mes_numero before to_datetime (has NaN): {df_mensal['mes_numero'].isnull().any()}")
        print(f"[DEBUG] Unique mes_numero values: {df_mensal['mes_numero'].unique()}")
        print(f"[DEBUG] Unique mes_numero as str: {df_mensal['mes_numero'].astype(str).unique()}")
        print(f"[DEBUG] Example data string for to_datetime: {(df_mensal['ano'].astype(str) + '-' + df_mensal['mes_numero'].astype(str) + '-01').iloc[0]}")

        df_mensal['data'] = pd.to_datetime(df_mensal['ano'].astype(str) + '-' + df_mensal['mes_numero'].astype(str) + '-01')
        df_mensal = df_mensal.sort_values('data').set_index('data')

        # Calcular média móvel
        df_mensal['media_movel_receita'] = df_mensal['receita_total'].rolling(window=janela_media_movel, min_periods=1).mean()

        ultima_data = df_mensal.index.max()
        ultimos_valores_validos = df_mensal['media_movel_receita'].dropna()

        if ultimos_valores_validos.empty:
            print("[AnalisadorComProjecao] Não há dados suficientes para calcular a média móvel.")
            return

        ultima_media_movel = ultimos_valores_validos.iloc[-1]

        # Projeção
        self.projecao_receita['futuro'] = {}
        for i in range(1, meses_a_projetar + 1):
            data_projecao = ultima_data + pd.DateOffset(months=i)
            mes_nome_proj = mes_map.get(data_projecao.month) # Usar mes_map do escopo global
            self.projecao_receita['futuro'][f"{mes_nome_proj}/{data_projecao.year}"] = ultima_media_movel
        print(f"[AnalisadorComProjecao] Projeção de receita para {meses_a_projetar} meses realizada.")

    def enriquecer_com_metas(self, df_metas_regiao):
        """Enrichece os dados com metas por região através de um merge."""
        if not self.metricas or 'por_regiao' not in self.metricas:
            print("[AnalisadorComProjecao] Erro: Métricas por região não disponíveis para enriquecimento.")
            return

        self.df_enriquecido_metas = pd.merge(
            self.metricas['por_regiao'],
            df_metas_regiao,
            on='regiao',
            how='left'
        )
        self.df_enriquecido_metas['meta_receita'] = self.df_enriquecido_metas['meta_receita'].fillna(0) # Tratar regiões sem meta
        print("[AnalisadorComProjecao] DataFrame enriquecido com metas por região.")

    def calcular_percentis(self):
        """Calcula os percentis (quartis) da receita total."""
        if self.df_limpo is None or 'receita_total' not in self.df_limpo.columns:
            print("[AnalisadorComProjecao] Erro: DataFrame limpo ou coluna 'receita_total' não disponível para percentis.")
            return

        receitas = self.df_limpo['receita_total'].to_numpy()
        self.percentis_receita = {
            'Q1': np.percentile(receitas, 25),
            'Mediana': np.percentile(receitas, 50),
            'Q3': np.percentile(receitas, 75)
        }
        print("[AnalisadorComProjecao] Percentis da receita total calculados.")

    def visualizar_adicional(self):
        """Gera gráficos adicionais: histograma e barras agrupadas para metas."""
        if self.df_limpo is None or self.df_enriquecido_metas is None:
            print("[AnalisadorComProjecao] Erro: Dados não disponíveis para visualizações adicionais.")
            return

        print("[AnalisadorComProjecao] Gerando gráficos adicionais...")

        # Histograma da Receita Total
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.histplot(self.df_limpo['receita_total'], bins=30, kde=True, ax=ax, color='lightcoral')
        ax.set_title('Distribuição da Receita Total por Item')
        ax.set_xlabel('Receita Total (R$)')
        ax.set_ylabel('Frequência')
        plt.tight_layout()
        plt.savefig("outputs/graficos/histograma_receita.png", dpi=150)
        plt.close()
        print("  Gráfico 'histograma_receita.png' gerado.")

        # Gráfico de Barras Agrupadas: Receita Real vs. Meta por Região
        fig, ax = plt.subplots(figsize=(12, 7))
        df_plot = self.df_enriquecido_metas.melt(id_vars='regiao', value_vars=['receita_total', 'meta_receita'],
                                             var_name='Tipo_Receita', value_name='Valor')
        sns.barplot(data=df_plot, x='regiao', y='Valor', hue='Tipo_Receita',
                    palette={'receita_total': 'teal', 'meta_receita': 'lightgray'}, ax=ax)
        ax.set_title('Receita Total Real vs. Meta por Região')
        ax.set_xlabel('Região')
        ax.set_ylabel('Valor (R$)')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title='Tipo de Receita')
        plt.tight_layout()
        plt.savefig("outputs/graficos/receita_vs_meta_regiao.png", dpi=150)
        plt.close()
        print("  Gráfico 'receita_vs_meta_regiao.png' gerado.")
        print("[AnalisadorComProjecao] Gráficos adicionais gerados e salvos.")

    def analisar(self):
        """Sobrescreve para incluir projeção, enriquecimento e percentis."""
        super().analisar()
        self.projetar_tendencia()
        self.enriquecer_com_metas(df_metas) # Usa o df_metas global
        self.calcular_percentis()

    def visualizar(self):
        """Sobrescreve para incluir visualizações adicionais."""
        super().visualizar()
        self.visualizar_adicional()

    def resumo(self):
        """Sobrescreve para incluir resumo da projeção e percentis."""
        super().resumo()
        if self.projecao_receita:
            print("\n--- Projeção de Receita (Média Móvel) ---")
            for mes, valor in self.projecao_receita['futuro'].items():
                print(f"  {mes}: R$ {valor:,.2f}")

        if self.percentis_receita:
            print("\n--- Percentis da Receita Total ---")
            for p, valor in self.percentis_receita.items():
                print(f"  {p}: R$ {valor:,.2f}")

        if self.df_enriquecido_metas is not None:
            print("\n--- Receita Real vs. Meta por Região ---")
            display(self.df_enriquecido_metas[['regiao', 'receita_total', 'meta_receita']])


### 10.3 - Demonstração da Classe `AnalisadorComProjecao`

Vamos agora executar o fluxo completo usando a nova subclasse para ver todas as funcionalidades avançadas em ação.

In [65]:
# Instanciar a subclasse
analisador_avancado = AnalisadorComProjecao(caminho_arquivo="vendas.csv")

# Executar o fluxo completo
print("\n" + "="*60)
print("          DEMONSTRAÇÃO DO ANALISADOR COM PROJEÇÃO")
print("="*60)

analisador_avancado.carregar()
analisador_avancado.limpar()
analisador_avancado.transformar()
analisador_avancado.analisar()
analisador_avancado.visualizar()
analisador_avancado.exportar_resultados() # Exporta os resultados (da classe base)
analisador_avancado.resumo() # Inclui o resumo das novas funcionalidades

print("\n[CONCLUÍDO] Demonstração da classe `AnalisadorComProjecao` finalizada com sucesso.")
print("=" * 60)



          DEMONSTRAÇÃO DO ANALISADOR COM PROJEÇÃO
[Analisador] 200 registros brutos lidos.
[Analisador] 183 registros limpos após processamento.
[Analisador] Colunas derivadas criadas.
[Analisador] Colunas 'receita_em_milhares' e 'perfil_volume' criadas usando `processar_coluna`.
[Analisador] Métricas agregadas calculadas.
[Analisador] Clientes segmentados.
Array NumPy 'receitas' criado com 183 elementos.
Array 'receitas_escalonadas' (0-1) criado usando broadcasting.
Filtragem booleana aplicada: 70 vendas acima da média.
[Analisador] Estatísticas NumPy calculadas.
[DEBUG] mes_numero before to_datetime (has NaN): False
[DEBUG] Unique mes_numero values: [ 4  8 12  2  1  7  6  5  3 11 10  9]
[DEBUG] Unique mes_numero as str: ['4' '8' '12' '2' '1' '7' '6' '5' '3' '11' '10' '9']
[DEBUG] Example data string for to_datetime: 2025-4-01
[AnalisadorComProjecao] Projeção de receita para 2 meses realizada.
[AnalisadorComProjecao] DataFrame enriquecido com metas por região.
[AnalisadorComProjecao]

,count
segmento,
Ouro,34
Prata,10
Bronze,6



--- Estatísticas NumPy da Receita ---
Média: R$ 7,051.07
Mediana: R$ 3,330.66
Desvio Padrão: R$ 7,809.92
Número de vendas acima da média: 70

         FIM DO RESUMO

--- Projeção de Receita (Média Móvel) ---
  Janeiro/2026: R$ 125,308.02
  Fevereiro/2026: R$ 125,308.02

--- Percentis da Receita Total ---
  Q1: R$ 1,193.40
  Mediana: R$ 3,330.66
  Q3: R$ 11,321.04

--- Receita Real vs. Meta por Região ---


,regiao,receita_total,meta_receita
0,Nordeste,366321.23,300000
1,Norte,299917.37,80000
2,Centro-Oeste,244586.97,100000
3,Sul,228249.66,180000
4,Sudeste,151271.07,350000



[CONCLUÍDO] Demonstração da classe `AnalisadorComProjecao` finalizada com sucesso.


# SalesInsight PY

## Sobre o projeto
Análise e visualização de dados de vendas desenvolvida em Python. O projeto carrega, limpa, transforma, agrega e visualiza um dataset de vendas, gerando métricas por período, produto, categoria e região, além de uma segmentação de clientes por faixa de gasto e projeção de tendências futuras. Também demonstra conceitos de Orientação a Objetos com herança e cálculos de percentis.

## O que o projeto analisa
- Receita total e volume de vendas por mês e por trimestre
- Top produtos e categorias por receita
- Desempenho por região
- Segmentação de clientes por nível de gasto (Bronze, Prata, Ouro)
- Relação entre quantidade vendida e receita por transação
- Projeção de receita futura com média móvel
- Percentis (quartis) da receita total
- Relatório de limpeza de dados
- Exportação de relatórios em CSV e JSON e de gráficos em PNG

## Conceitos aplicados (Módulo 01 - Semanas 01 a 08)
- Lógica de programação: variáveis, tipos, operadores, condicionais
- Estruturas de dados: listas, tuplas, dicionários e compostas
- Funções: parâmetros, retorno, docstrings, lambda, ordem superior
- Leitura e escrita de arquivos CSV e JSON
- Módulo `datetime` e expressões regulares (`re`)
- Pandas: Series, DataFrames, filtros, groupby, transformações, merge, pivot
- NumPy: arrays, operações vetorizadas, broadcasting, percentis
- Matplotlib e Seaborn: linha, barra, dispersão, histograma, subplots, gráficos de barra agrupados, exportação
- Introdução a classes: construtor, atributos e métodos, herança (`super()`)
- Git e GitHub: branches, commits e GitFlow simplificado

## Como executar
### Google Colab (recomendado)
1.  **Abra o Notebook:** Faça o upload do arquivo `.ipynb` para o Google Colab ou abra-o diretamente se já estiver no Drive.
2.  **Execute todas as células:** Vá em `Ambiente de execução` (Runtime) no menu superior e selecione `Executar tudo` (Run all). O notebook irá gerar o dataset fictício, processar os dados, gerar os gráficos e exportar os resultados para a pasta `outputs/`.

### Localmente com VS Code
1.  Instale o Python 3.10+ e o VS Code.
2.  Instale as dependências:
    ```bash
    pip install pandas numpy matplotlib seaborn
    ```
3.  Execute o script Python (se você exportou o notebook como .py):
    ```bash
    python salesinsight.py
    ```

## Estrutura do projeto
```
salesinsight-py/
|-- salesinsight.ipynb  # Notebook principal com o fluxo completo
|-- vendas.csv          # Dataset (gerado pelo notebook ou externo)
|-- README.md           # Este arquivo
|-- outputs/
|    |-- metricas_por_mes.csv
|    |-- segmentacao_clientes.csv
|    |-- estatisticas_gerais.json
|    |-- graficos/
|    |    |-- receita_por_mes.png
|    |    |-- top_produtos.png
|    |    |-- quantidade_vs_receita.png
|    |    |-- painel_resumo.png
|    |    |-- histograma_receita.png
|    |    |-- receita_vs_meta_regiao.png
```

## Decisões técnicas
Uma decisão técnica importante foi remover os registros com `data_venda` inválida e valores nulos críticos (`quantidade`, `preco_unitario`). Embora fosse possível tentar imputar esses valores, a remoção foi escolhida para garantir a **integridade dos cálculos de receita e análises temporais**, evitando a introdução de vieses ou distorções causadas por dados sintéticos ou incompletos. A precisão nessas métricas é fundamental para as decisões de negócio.

## Ferramentas utilizadas
- Python 3.10+
- Google Colab / VS Code
- Bibliotecas: `pandas`, `numpy`, `matplotlib`, `seaborn`, `re`, `json`, `datetime`, `os`, `random`
- GitHub para versionamento

## Vídeo de demonstração
[Inserir o link do Google Drive ou do YouTube aqui - em breve!]
